In [7]:
from pathlib import Path

FASTFIT_REPO = Path("/home/ester/valid_description/FastFit").resolve()
print("FASTFIT_REPO =", FASTFIT_REPO)
print("exists =", FASTFIT_REPO.exists())

FASTFIT_REPO = /home/ester/valid_description/FastFit
exists = True


In [1]:
# ============================================================
# CLEAN FULL ONE-CELL DEMO
# specific test case: 224188768
# use ONLY:
#   - /home/ester/valid_description/new_polyvore_outfit_titles.json
#   - /home/ester/valid_description/fashionclip_data/encoded_NewoutfitUrlTitle_en_fashionClip.pkl
# ============================================================

from pathlib import Path
import os
import sys
import json
import random
import types
from contextlib import nullcontext
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 0) 路徑與參數
# ------------------------------------------------------------
PROJECT_ROOT = Path("/home/ester/valid_description/text-conditioned-outfit-recommendation").resolve()
POLYVORE_DATA_PARENT = Path("/home/ester/valid_description/polyvore_data")
POLYVORE_OUTFIT_ROOT = POLYVORE_DATA_PARENT / "polyvore_outfits"
POLYVORE_SPLIT = "disjoint"

CATEGORY_CSV = POLYVORE_OUTFIT_ROOT / "categories.csv"
TEST_JSON = POLYVORE_OUTFIT_ROOT / POLYVORE_SPLIT / "test.json"
NEW_TITLE_JSON = Path("/home/ester/valid_description/new_polyvore_outfit_titles.json")
OUTFIT_FEAT_PATH = Path("/home/ester/valid_description/fashionclip_data/encoded_NewoutfitUrlTitle_en_fashionClip.pkl")

FASHIONCLIP_DIR = Path("/home/ester/valid_description/fashionclip_data")
IMG_FEAT_PATH = FASHIONCLIP_DIR / "img_feats_fashionClip.pkl"
TXT_FEAT_PATH = FASHIONCLIP_DIR / "encoded_title_description_distiluse-base-multilingual-cased-v2.pkl"
CATEGORY_FEAT_PATH = FASHIONCLIP_DIR / "encoded_category_distiluse-base-multilingual-cased-v2.pkl"

PERSON_IMAGE = PROJECT_ROOT / "model_image.png"
CKPT_PATH = PROJECT_ROOT / "experiments" / "cir_new_seed1" / "ckpt.pt"

FASTFIT_REPO = Path("/home/ester/valid_description/FastFit").resolve()

CASE_SET_ID = "224188768"
TOPK_PER_STEP = 20
SAMPLING_TOPK = None
NORMALIZE_OUTFIT_TEXT = False
RUN_FASTFIT = True

OUTPUT_DIR = PROJECT_ROOT / "jupyter_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = OUTPUT_DIR / f"case_{CASE_SET_ID}_clean"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BANK_CACHE = OUTPUT_DIR / "candidate_bank_new_seed1_fg_based_v3.pt"

# ------------------------------------------------------------
# 1) 基本檢查
# ------------------------------------------------------------
for p in [CATEGORY_CSV, TEST_JSON, NEW_TITLE_JSON, OUTFIT_FEAT_PATH, IMG_FEAT_PATH, TXT_FEAT_PATH, CATEGORY_FEAT_PATH, CKPT_PATH, PERSON_IMAGE]:
    assert Path(p).exists(), f"找不到檔案: {p}"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("CASE_SET_ID =", CASE_SET_ID)
print("NEW_TITLE_JSON =", NEW_TITLE_JSON)
print("OUTFIT_FEAT_PATH =", OUTFIT_FEAT_PATH)
print("FASTFIT_REPO exists =", FASTFIT_REPO.exists())

# ------------------------------------------------------------
# 2) 相容 shim
# ------------------------------------------------------------
if "attn_utils" not in sys.modules:
    attn_utils = types.ModuleType("attn_utils")

    class DecoderLayerWithCrossAttn(nn.TransformerDecoderLayer):
        def __init__(self, *args, **kwargs):
            kwargs.setdefault("batch_first", True)
            super().__init__(*args, **kwargs)

        def forward(
            self,
            tgt,
            memory=None,
            tgt_mask=None,
            memory_mask=None,
            tgt_key_padding_mask=None,
            memory_key_padding_mask=None,
            tgt_is_causal=False,
            memory_is_causal=False,
            **kwargs,
        ):
            if memory is None:
                memory = tgt
            return super().forward(
                tgt=tgt,
                memory=memory,
                tgt_mask=tgt_mask,
                memory_mask=memory_mask,
                tgt_key_padding_mask=tgt_key_padding_mask,
                memory_key_padding_mask=memory_key_padding_mask,
                tgt_is_causal=tgt_is_causal,
                memory_is_causal=memory_is_causal,
            )

    attn_utils.DecoderLayerWithCrossAttn = DecoderLayerWithCrossAttn
    sys.modules["attn_utils"] = attn_utils

if "ipdb" not in sys.modules:
    ipdb = types.ModuleType("ipdb")
    def _noop(*args, **kwargs):
        return None
    ipdb.set_trace = _noop
    ipdb.iex = _noop
    ipdb.launch_ipdb_on_exception = lambda: nullcontext()
    sys.modules["ipdb"] = ipdb

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ------------------------------------------------------------
# 3) import project modules
# ------------------------------------------------------------
from dataset import UIUCPolyvoreRetrievalDataset
from outfit_transformer import OutfitTransformerRetrieval, LinearImageEncoder, LinearTextEncoder
import config.cir_cond_hardneg as cfg
from config.cir_cond_hardneg import *

# ------------------------------------------------------------
# 4) utility
# ------------------------------------------------------------
plt.rcParams["figure.dpi"] = 140
plt.rcParams["font.family"] = "DejaVu Sans"

def set_seed(seed=1):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(1)

device = torch.device(device if isinstance(device, str) else device)
ptdtype = {
    "float32": torch.float32,
    "bfloat16": torch.bfloat16,
    "float16": torch.float16,
}[dtype]
ctx = nullcontext() if device_type == "cpu" else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

IMAGE_ROOTS = [POLYVORE_OUTFIT_ROOT / "images"]

def canon_category_id(x):
    if x is None:
        return None
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return None
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
        return s
    except Exception:
        return s

def first_existing_image(item_id: str):
    item_id = str(item_id)
    for root in IMAGE_ROOTS:
        for ext in [".jpg", ".jpeg", ".png", ".webp"]:
            p = Path(root) / f"{item_id}{ext}"
            if p.exists():
                return p
    return None

def normalize_text_value(v):
    if v is None:
        return ""
    if isinstance(v, str):
        return v.strip()
    if isinstance(v, list):
        return " | ".join(str(x).strip() for x in v if str(x).strip())
    if isinstance(v, dict):
        parts = []
        for k in ["url_name", "url", "title"]:
            if k in v and str(v[k]).strip():
                parts.append(str(v[k]).strip())
        return " ".join(parts).strip()
    return str(v).strip()

def open_image_or_none(path_or_item_id):
    p = Path(str(path_or_item_id))
    if p.exists():
        try:
            return Image.open(p).convert("RGB")
        except Exception:
            return None

    meta = item_meta.get(str(path_or_item_id))
    if meta is None or meta.get("image_path") is None:
        return None

    p = Path(meta["image_path"])
    if not p.exists():
        return None

    try:
        return Image.open(p).convert("RGB")
    except Exception:
        return None

def show_ref_row(refs, title, fig_scale=2.8):
    labels = ["upper", "lower", "overall", "shoe", "bag"]
    fig, axes = plt.subplots(1, len(labels), figsize=(fig_scale * len(labels), fig_scale))
    for ax, label in zip(axes, labels):
        path = refs.get(label)
        img = open_image_or_none(path) if path is not None else None
        if img is not None:
            ax.imshow(img)
        else:
            ax.set_facecolor("#eeeeee")
            ax.text(0.5, 0.5, "N/A", ha="center", va="center", fontsize=10, transform=ax.transAxes)
        ax.set_title(label, fontsize=10)
        ax.axis("off")
    fig.suptitle(title, fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# 5) category mapping
# ------------------------------------------------------------
def load_category_major_map(csv_path):
    df = pd.read_csv(csv_path, header=None)
    major_to_fastfit = {
        "tops": "upper",
        "outerwear": "upper",
        "bottoms": "lower",
        "all-body": "overall",
        "shoes": "shoe",
        "bags": "bag",
    }

    result = {}
    for _, row in df.iterrows():
        cid = canon_category_id(row[0])
        if cid is None or cid in result:
            continue

        fine_category = str(row[1]).strip().lower() if pd.notna(row[1]) else ""
        major_category = str(row[2]).strip().lower() if pd.notna(row[2]) else ""

        result[cid] = {
            "fine_category": fine_category,
            "major_category": major_category,
            "fastfit_category": major_to_fastfit.get(major_category, None),
        }
    return result

CATEGORY_INFO = load_category_major_map(CATEGORY_CSV)

# ------------------------------------------------------------
# 6) dataset + model
# ------------------------------------------------------------
normalize = torchvision.transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)
transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((img_size, img_size)),
    torchvision.transforms.ToTensor(),
    normalize,
])

base_dat = UIUCPolyvoreRetrievalDataset(
    datadir=str(POLYVORE_DATA_PARENT),
    txt_feat_path=str(TXT_FEAT_PATH),
    img_feat_path=str(IMG_FEAT_PATH),
    category_feat_path=str(CATEGORY_FEAT_PATH),
    outfit_feat_path=str(OUTFIT_FEAT_PATH),
    polyvore_split=POLYVORE_SPLIT,
    split="test",
    max_item_len=max_item_len,
    transform=transform,
    sample_hard_negative=False,
)

model = OutfitTransformerRetrieval(
    img_encoder=LinearImageEncoder(img_inp_size=img_inp_size, img_emb_size=img_emb_size),
    text_encoder=LinearTextEncoder(txt_inp_size=txt_inp_size, txt_emb_size=txt_emb_size),
    outfit_txt_encoder=LinearTextEncoder(
        txt_inp_size=outfit_txt_inp_size,
        txt_emb_size=txt_emb_size + img_emb_size,
    ) if use_outfit_txt else None,
    nhead=nhead,
    num_layers=num_layers,
    margin=margin,
    target_item_info="category",
    use_outfit_txt=use_outfit_txt,
)

checkpoint = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(checkpoint["model"], strict=True)
model = model.to(device)
model.eval()

print("checkpoint loaded:", CKPT_PATH)

# ------------------------------------------------------------
# 7) text encoder
# ------------------------------------------------------------
from fashion_clip.fashion_clip import FashionCLIP
fclip = FashionCLIP("fashion-clip")

def encode_outfit_text(text: str, normalize=False):
    arr = fclip.encode_text([text], batch_size=1)[0].astype(np.float32)
    if normalize:
        norm = np.linalg.norm(arr)
        if norm > 0:
            arr = arr / norm
    return torch.tensor(arr, dtype=torch.float32, device=device)

# ------------------------------------------------------------
# 8) item_meta from fg
# ------------------------------------------------------------
def build_item_meta_from_fg(base_dat):
    item_meta = {}
    for item_id in base_dat.category_feats.keys():
        item_id = str(item_id)
        fg_id = canon_category_id(base_dat.im2fg.get(item_id, None))
        cat_info = CATEGORY_INFO.get(fg_id)

        if cat_info is None:
            fine_category = None
            major_category = None
            fastfit_category = None
        else:
            fine_category = cat_info["fine_category"]
            major_category = cat_info["major_category"]
            fastfit_category = cat_info["fastfit_category"]

        image_path = first_existing_image(item_id)
        item_meta[item_id] = {
            "item_id": item_id,
            "fg_id": fg_id,
            "fine_category": fine_category,
            "major_category": major_category,
            "fastfit_category": fastfit_category,
            "image_path": str(image_path) if image_path else None,
            "title": "",
        }
    return item_meta

item_meta = build_item_meta_from_fg(base_dat)

# ------------------------------------------------------------
# 9) candidate bank + category condition bank
# ------------------------------------------------------------
def extract_item_embedding(item_id: str):
    with torch.no_grad(), ctx:
        emb = base_dat.extract_emb({"item_id": str(item_id)}, model, device, ctx, to_numpy=False)
    return emb.detach().cpu().float()

def build_candidate_bank(item_meta, cache_path=None):
    if cache_path is not None and Path(cache_path).exists():
        print("load candidate bank from cache:", cache_path)
        return torch.load(cache_path, map_location="cpu")

    category_to_ids = defaultdict(list)
    for item_id, meta in item_meta.items():
        ff_cat = meta.get("fastfit_category")
        if ff_cat is None:
            continue
        if meta.get("image_path") is None:
            continue
        if str(item_id) not in base_dat.category_feats:
            continue
        category_to_ids[ff_cat].append(str(item_id))

    candidate_bank = {}
    for ff_cat, ids in category_to_ids.items():
        emb_list = []
        good_ids = []
        for item_id in tqdm(ids, desc=f"build bank: {ff_cat}"):
            try:
                emb = extract_item_embedding(item_id)
                emb_list.append(emb)
                good_ids.append(item_id)
            except Exception:
                pass
        if good_ids:
            candidate_bank[ff_cat] = {
                "ids": good_ids,
                "embs": torch.cat(emb_list, dim=0),
            }

    if cache_path is not None:
        torch.save(candidate_bank, cache_path)
    return candidate_bank

candidate_bank = build_candidate_bank(item_meta, BANK_CACHE)

def build_category_condition_bank(candidate_bank):
    cond_bank = {}
    for ff_cat, pack in candidate_bank.items():
        vecs = []
        for item_id in pack["ids"][:512]:
            if str(item_id) in base_dat.category_feats:
                vec = torch.tensor(base_dat.category_feats[str(item_id)], dtype=torch.float32)
                vecs.append(vec)
        if vecs:
            cond_bank[ff_cat] = torch.stack(vecs, dim=0).mean(dim=0).to(device)
    return cond_bank

category_condition_bank = build_category_condition_bank(candidate_bank)
print("category condition bank keys:", list(category_condition_bank.keys()))

# ------------------------------------------------------------
# 10) keep0 generation
# ------------------------------------------------------------
def infer_empty_shapes():
    any_item_id = next(iter(base_dat.category_feats.keys()))
    imgs, txts, _ = base_dat.load_outfit({"items": [{"item_id": str(any_item_id)}]})
    padded_imgs, padded_txts, padded_mask = base_dat.pad_imgs_and_txts(imgs, txts)
    return padded_imgs.shape[-1], padded_txts.shape[-1], padded_mask.dtype

EMPTY_IMG_DIM, EMPTY_TXT_DIM, MASK_DTYPE = infer_empty_shapes()

def prepare_partial_inputs(partial_item_ids):
    if len(partial_item_ids) == 0:
        partial_imgs = torch.zeros((max_item_len, EMPTY_IMG_DIM), dtype=torch.float32, device=device)
        partial_txts = torch.zeros((max_item_len, EMPTY_TXT_DIM), dtype=torch.float32, device=device)
        mask = torch.ones((max_item_len,), dtype=torch.bool, device=device)
        return partial_imgs, partial_txts, mask

    items = [{"item_id": str(iid)} for iid in partial_item_ids]
    partial_imgs, partial_txts, _ = base_dat.load_outfit({"items": items})
    partial_imgs, partial_txts, mask = base_dat.pad_imgs_and_txts(partial_imgs, partial_txts)
    return partial_imgs.to(device), partial_txts.to(device), mask.to(device)

def rank_candidates_for_category(logits, ff_cat, exclude_ids=None, topk=20):
    exclude_ids = set(str(x) for x in (exclude_ids or []))
    pack = candidate_bank[ff_cat]
    sims = F.cosine_similarity(logits.cpu(), pack["embs"], dim=1)
    order = torch.argsort(sims, descending=True).tolist()

    ranked = []
    for idx in order:
        item_id = str(pack["ids"][idx])
        if item_id in exclude_ids:
            continue
        ranked.append({
            "item_id": item_id,
            "score": float(sims[idx].item()),
            "fastfit_category": ff_cat,
            "image_path": item_meta[item_id]["image_path"],
        })
        if len(ranked) >= topk:
            break
    return ranked

def choose_candidate(ranked_items, sampling_topk=None):
    if not ranked_items:
        raise ValueError("沒有可選 candidate")
    if sampling_topk is None or sampling_topk <= 1:
        return ranked_items[0]

    top_items = ranked_items[:sampling_topk]
    scores = np.array([max(x["score"], 1e-8) for x in top_items], dtype=np.float64)
    probs = scores / scores.sum()
    idx = np.random.choice(len(top_items), p=probs)
    return top_items[idx]

def generate_outfit_keep0(outfit_text, schema, topk_per_step=20, sampling_topk=None, verbose=True):
    outfit_txt = encode_outfit_text(outfit_text, normalize=NORMALIZE_OUTFIT_TEXT)
    partial_item_ids = []
    step_results = []

    for step_idx, ff_cat in enumerate(schema, start=1):
        partial_imgs, partial_txts, mask = prepare_partial_inputs(partial_item_ids)
        positive_category = category_condition_bank[ff_cat]

        with torch.no_grad(), ctx:
            out = model(
                partial_imgs.unsqueeze(0),
                partial_txts.unsqueeze(0),
                mask.unsqueeze(0),
                positive_category=positive_category.unsqueeze(0),
                outfit_txt=outfit_txt.unsqueeze(0),
            )
            logits = out["logits"].detach().cpu().float()

        ranked = rank_candidates_for_category(logits, ff_cat, exclude_ids=partial_item_ids, topk=topk_per_step)
        chosen = choose_candidate(ranked, sampling_topk=sampling_topk)
        partial_item_ids.append(chosen["item_id"])

        step_results.append({
            "step": step_idx,
            "target_fastfit_category": ff_cat,
            "chosen_item_id": chosen["item_id"],
            "chosen_score": chosen["score"],
            "image_path": chosen["image_path"],
        })

        if verbose:
            print(f"[step {step_idx}] {ff_cat:>7s} -> {chosen['item_id']} | score={chosen['score']:.4f}")

    return {
        "outfit_text": outfit_text,
        "schema": schema,
        "items": step_results,
        "mean_score": float(np.mean([x["chosen_score"] for x in step_results])),
    }

# ------------------------------------------------------------
# 11) load NEW descriptions only
# ------------------------------------------------------------
with open(NEW_TITLE_JSON, "r", encoding="utf-8") as f:
    new_title_map_raw = json.load(f)

new_title_map = {}
for sid, value in new_title_map_raw.items():
    new_title_map[str(sid)] = normalize_text_value(value)

assert CASE_SET_ID in new_title_map, f"new_polyvore_outfit_titles.json 找不到 set_id={CASE_SET_ID}"
selected_text = new_title_map[CASE_SET_ID]

with open(TEST_JSON, "r", encoding="utf-8") as f:
    test_outfits = json.load(f)

test_outfit_map = {str(x["set_id"]): x for x in test_outfits}
assert CASE_SET_ID in test_outfit_map, f"test.json 找不到 set_id={CASE_SET_ID}"
selected_outfit = test_outfit_map[CASE_SET_ID]

print("Selected set_id =", CASE_SET_ID)
print("Query text =", selected_text)

# ------------------------------------------------------------
# 12) GT refs + GT schema
# ------------------------------------------------------------
def extract_fastfit_refs_from_outfit(outfit):
    refs = {"upper": None, "lower": None, "overall": None, "shoe": None, "bag": None}
    for item in outfit.get("items", []):
        item_id = str(item.get("item_id"))
        meta = item_meta.get(item_id, {})
        ff_cat = meta.get("fastfit_category")
        img_path = meta.get("image_path")
        if ff_cat in refs and refs[ff_cat] is None and img_path is not None:
            refs[ff_cat] = img_path
    return refs

def infer_schema_from_refs(refs):
    return [label for label in ["upper", "lower", "overall", "shoe", "bag"] if refs.get(label) is not None]

selected_gt_refs = extract_fastfit_refs_from_outfit(selected_outfit)
gt_schema = infer_schema_from_refs(selected_gt_refs)

if "overall" in gt_schema and ("upper" in gt_schema or "lower" in gt_schema):
    gt_schema = [x for x in gt_schema if x != "overall"]

print("GT refs =", selected_gt_refs)
print("GT schema =", gt_schema)

# ------------------------------------------------------------
# 13) force generation with GT schema
# ------------------------------------------------------------
forced_result = generate_outfit_keep0(
    outfit_text=selected_text,
    schema=gt_schema,
    topk_per_step=TOPK_PER_STEP,
    sampling_topk=SAMPLING_TOPK,
    verbose=True,
)

forced_generated_refs = {"upper": None, "lower": None, "overall": None, "shoe": None, "bag": None}
for row in forced_result["items"]:
    forced_generated_refs[row["target_fastfit_category"]] = row["image_path"]

print("\nForced generated refs:")
print(forced_generated_refs)

show_ref_row(selected_gt_refs, f"Ground Truth from TEST set | set_id={CASE_SET_ID}")
show_ref_row(forced_generated_refs, f"Generated with GT Schema | query={selected_text}")

# ------------------------------------------------------------
# 14) FastFit (optional)
# ------------------------------------------------------------
if RUN_FASTFIT:
    if not FASTFIT_REPO.exists():
        print(f"[SKIP] 找不到 FastFit repo: {FASTFIT_REPO}")
    else:
        if str(FASTFIT_REPO) not in sys.path:
            sys.path.insert(0, str(FASTFIT_REPO))

        try:
            from app import FastFitDemo

            demo = FastFitDemo(
                base_model_path=str(FASTFIT_REPO / "Models" / "FastFit-MR-1024"),
                util_model_path=str(FASTFIT_REPO / "Models" / "Human-Toolkit"),
                mixed_precision="bf16",
                device="cuda" if torch.cuda.is_available() else "cpu",
            )

            person_img = Image.open(PERSON_IMAGE).convert("RGB")
            upper_img = Image.open(forced_generated_refs["upper"]).convert("RGB") if forced_generated_refs["upper"] else None
            lower_img = Image.open(forced_generated_refs["lower"]).convert("RGB") if forced_generated_refs["lower"] else None
            dress_img = Image.open(forced_generated_refs["overall"]).convert("RGB") if forced_generated_refs["overall"] else None
            shoe_img  = Image.open(forced_generated_refs["shoe"]).convert("RGB") if forced_generated_refs["shoe"] else None
            bag_img   = Image.open(forced_generated_refs["bag"]).convert("RGB") if forced_generated_refs["bag"] else None

            result_img, status = demo.generate_image(
                person_img=person_img,
                upper_img=upper_img,
                lower_img=lower_img,
                dress_img=dress_img,
                shoe_img=shoe_img,
                bag_img=bag_img,
                ref_height=768,
                num_inference_steps=30,
                guidance_scale=2.5,
                use_square_mask=False,
                seed=42,
                enable_pose=True,
            )

            print("FastFit status:", status)

            if result_img is not None:
                fastfit_out_path = OUT_DIR / f"fastfit_case_{CASE_SET_ID}.png"
                result_img.save(fastfit_out_path)
                print("saved:", fastfit_out_path)

                plt.figure(figsize=(5, 7))
                plt.imshow(result_img)
                plt.title(f"FastFit Result | set_id={CASE_SET_ID}")
                plt.axis("off")
                plt.show()
            else:
                print("FastFit 沒有產生有效結果")

        except Exception as e:
            print("\n[FastFit skipped]")
            print("原因：", repr(e))
            print("目前 GT 與 generated 商品列已完成；FastFit 依賴補齊後再重跑這格即可。")
else:
    print("RUN_FASTFIT=False，目前只顯示 GT 與 Generated 商品列。")

PROJECT_ROOT = /home/ester/valid_description/text-conditioned-outfit-recommendation
CASE_SET_ID = 224188768
NEW_TITLE_JSON = /home/ester/valid_description/new_polyvore_outfit_titles.json
OUTFIT_FEAT_PATH = /home/ester/valid_description/fashionclip_data/encoded_NewoutfitUrlTitle_en_fashionClip.pkl
FASTFIT_REPO exists = True


2026-04-02 01:44:07.945462: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-02 01:44:07.953869: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775065447.962353    5554 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775065447.965178    5554 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775065447.973113    5554 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

checkpoint loaded: /home/ester/valid_description/text-conditioned-outfit-recommendation/experiments/cir_new_seed1/ckpt.pt


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


load candidate bank from cache: /home/ester/valid_description/text-conditioned-outfit-recommendation/jupyter_outputs/candidate_bank_new_seed1_fg_based_v3.pt
category condition bank keys: ['lower', 'upper', 'shoe', 'bag', 'overall']
Selected set_id = 224188768
Query text = 26.2° C, Effortless Summer Wedding Style with a Pop of Citrus
GT refs = {'upper': None, 'lower': None, 'overall': '/home/ester/valid_description/polyvore_data/polyvore_outfits/images/209144226.jpg', 'shoe': '/home/ester/valid_description/polyvore_data/polyvore_outfits/images/208337886.jpg', 'bag': '/home/ester/valid_description/polyvore_data/polyvore_outfits/images/201548065.jpg'}
GT schema = ['overall', 'shoe', 'bag']


...

<Figure size 1960x392 with 5 Axes>

<Figure size 1960x392 with 5 Axes>

/home/ester/miniforge3/envs/minenv/lib/python3.12/site-packages/networkx/utils/backends.py:127: RuntimeWarning: Error encountered when loading info for backend cugraph: No module named '_nx_cugraph'
  backend_info.update(_get_backends("networkx.backend_info", load_and_call=True))
An error occurred while trying to fetch /home/ester/valid_description/FastFit/Models/FastFit-MR-1024: Error no file named diffusion_pytorch_model.safetensors found in directory /home/ester/valid_description/FastFit/Models/FastFit-MR-1024.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
/home/ester/miniforge3/envs/minenv/lib/python3.12/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0402 01:44:46.374000 5554 site-packages/torch/fx

FastFit status: success_generation
saved: /home/ester/valid_description/text-conditioned-outfit-recommendation/jupyter_outputs/case_224188768_clean/fastfit_case_224188768.png


<Figure size 700x980 with 1 Axes>

In [1]:
# ============================================================
# FULL ONE-CELL DEMO
# Keep0 / Keep1 / Keep2 + paper-style comparison + FastFit try-on
# Directly runnable in ONE cell
# ============================================================

from pathlib import Path
import os
import sys
import json
import random
import types
from contextlib import nullcontext
from collections import defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 0) paths / params
# ------------------------------------------------------------
PROJECT_ROOT = Path("/home/ester/valid_description/text-conditioned-outfit-recommendation").resolve()
POLYVORE_DATA_PARENT = Path("/home/ester/valid_description/polyvore_data")
POLYVORE_OUTFIT_ROOT = POLYVORE_DATA_PARENT / "polyvore_outfits"
POLYVORE_SPLIT = "disjoint"

CATEGORY_CSV = POLYVORE_OUTFIT_ROOT / "categories.csv"
TEST_JSON = POLYVORE_OUTFIT_ROOT / POLYVORE_SPLIT / "test.json"
NEW_TITLE_JSON = Path("/home/ester/valid_description/new_polyvore_outfit_titles.json")
OUTFIT_FEAT_PATH = Path("/home/ester/valid_description/fashionclip_data/encoded_NewoutfitUrlTitle_en_fashionClip.pkl")

FASHIONCLIP_DIR = Path("/home/ester/valid_description/fashionclip_data")
IMG_FEAT_PATH = FASHIONCLIP_DIR / "img_feats_fashionClip.pkl"
TXT_FEAT_PATH = FASHIONCLIP_DIR / "encoded_title_description_distiluse-base-multilingual-cased-v2.pkl"
CATEGORY_FEAT_PATH = FASHIONCLIP_DIR / "encoded_category_distiluse-base-multilingual-cased-v2.pkl"

PERSON_IMAGE = PROJECT_ROOT / "model_image.png"
CKPT_PATH = PROJECT_ROOT / "experiments" / "cir_new_seed1" / "ckpt.pt"

FASTFIT_REPO = Path("/home/ester/valid_description/FastFit").resolve()

CASE_SET_ID = "224188768"

TOPK_PER_STEP = 20
SAMPLING_TOPK = None
NORMALIZE_OUTFIT_TEXT = False
RUN_FASTFIT = True
SAVE_JSON = True

# 可手動指定要保留哪些類別；若設為 None，就自動用 prefix
# 例如 KEEP1_CATEGORIES = ["shoe"]
#     KEEP2_CATEGORIES = ["upper", "shoe"]
KEEP1_CATEGORIES = None
KEEP2_CATEGORIES = None
KEEP_MODE_IF_NOT_SPECIFIED = "prefix"   # "prefix" or "suffix"

# paper-style figure 只顯示這 5 個槽位
DISPLAY_ORDER = ["upper", "lower", "overall", "shoe", "bag"]

OUTPUT_DIR = PROJECT_ROOT / "jupyter_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = OUTPUT_DIR / f"case_{CASE_SET_ID}_keep012_full"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BANK_CACHE = OUTPUT_DIR / "candidate_bank_new_seed1_fg_based_v3.pt"

# ------------------------------------------------------------
# 1) basic checks
# ------------------------------------------------------------
for p in [
    CATEGORY_CSV, TEST_JSON, NEW_TITLE_JSON, OUTFIT_FEAT_PATH,
    IMG_FEAT_PATH, TXT_FEAT_PATH, CATEGORY_FEAT_PATH,
    CKPT_PATH, PERSON_IMAGE
]:
    assert Path(p).exists(), f"找不到檔案: {p}"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("CASE_SET_ID =", CASE_SET_ID)
print("OUT_DIR =", OUT_DIR)
print("FASTFIT_REPO exists =", FASTFIT_REPO.exists())

# ------------------------------------------------------------
# 2) compatibility shim
# ------------------------------------------------------------
if "attn_utils" not in sys.modules:
    attn_utils = types.ModuleType("attn_utils")

    class DecoderLayerWithCrossAttn(nn.TransformerDecoderLayer):
        def __init__(self, *args, **kwargs):
            kwargs.setdefault("batch_first", True)
            super().__init__(*args, **kwargs)

        def forward(
            self,
            tgt,
            memory=None,
            tgt_mask=None,
            memory_mask=None,
            tgt_key_padding_mask=None,
            memory_key_padding_mask=None,
            tgt_is_causal=False,
            memory_is_causal=False,
            **kwargs,
        ):
            if memory is None:
                memory = tgt
            return super().forward(
                tgt=tgt,
                memory=memory,
                tgt_mask=tgt_mask,
                memory_mask=memory_mask,
                tgt_key_padding_mask=tgt_key_padding_mask,
                memory_key_padding_mask=memory_key_padding_mask,
                tgt_is_causal=tgt_is_causal,
                memory_is_causal=memory_is_causal,
            )

    attn_utils.DecoderLayerWithCrossAttn = DecoderLayerWithCrossAttn
    sys.modules["attn_utils"] = attn_utils

if "ipdb" not in sys.modules:
    ipdb = types.ModuleType("ipdb")
    def _noop(*args, **kwargs):
        return None
    ipdb.set_trace = _noop
    ipdb.iex = _noop
    ipdb.launch_ipdb_on_exception = lambda: nullcontext()
    sys.modules["ipdb"] = ipdb

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ------------------------------------------------------------
# 3) import project modules
# ------------------------------------------------------------
from dataset import UIUCPolyvoreRetrievalDataset
from outfit_transformer import OutfitTransformerRetrieval, LinearImageEncoder, LinearTextEncoder
import config.cir_cond_hardneg as cfg
from config.cir_cond_hardneg import *

# ------------------------------------------------------------
# 4) utilities
# ------------------------------------------------------------
plt.rcParams["figure.dpi"] = 140
plt.rcParams["font.family"] = "DejaVu Sans"

def set_seed(seed=1):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(1)

device = torch.device(device if isinstance(device, str) else device)
ptdtype = {
    "float32": torch.float32,
    "bfloat16": torch.bfloat16,
    "float16": torch.float16,
}[dtype]
ctx = nullcontext() if device_type == "cpu" else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

IMAGE_ROOTS = [POLYVORE_OUTFIT_ROOT / "images"]

def canon_category_id(x):
    if x is None:
        return None
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return None
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
        return s
    except Exception:
        return s

def first_existing_image(item_id: str):
    item_id = str(item_id)
    for root in IMAGE_ROOTS:
        for ext in [".jpg", ".jpeg", ".png", ".webp"]:
            p = Path(root) / f"{item_id}{ext}"
            if p.exists():
                return p
    return None

def normalize_text_value(v):
    if v is None:
        return ""
    if isinstance(v, str):
        return v.strip()
    if isinstance(v, list):
        return " | ".join(str(x).strip() for x in v if str(x).strip())
    if isinstance(v, dict):
        parts = []
        for k in ["url_name", "url", "title"]:
            if k in v and str(v[k]).strip():
                parts.append(str(v[k]).strip())
        return " ".join(parts).strip()
    return str(v).strip()

def open_image_or_none(path_or_item_id):
    if path_or_item_id is None:
        return None

    p = Path(str(path_or_item_id))
    if p.exists():
        try:
            return Image.open(p).convert("RGB")
        except Exception:
            return None

    meta = item_meta.get(str(path_or_item_id))
    if meta is None or meta.get("image_path") is None:
        return None

    p = Path(meta["image_path"])
    if not p.exists():
        return None

    try:
        return Image.open(p).convert("RGB")
    except Exception:
        return None

def show_ref_row(refs, title, fig_scale=2.7):
    labels = DISPLAY_ORDER
    fig, axes = plt.subplots(1, len(labels), figsize=(fig_scale * len(labels), fig_scale))
    for ax, label in zip(axes, labels):
        path = refs.get(label)
        img = open_image_or_none(path) if path is not None else None
        if img is not None:
            ax.imshow(img)
        else:
            ax.set_facecolor("#eeeeee")
            ax.text(0.5, 0.5, "N/A", ha="center", va="center", fontsize=10, transform=ax.transAxes)
        ax.set_title(label, fontsize=10)
        ax.axis("off")
    fig.suptitle(title, fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# 5) category mapping
# ------------------------------------------------------------
def load_category_major_map(csv_path):
    df = pd.read_csv(csv_path, header=None)
    major_to_fastfit = {
        "tops": "upper",
        "outerwear": "upper",
        "bottoms": "lower",
        "all-body": "overall",
        "shoes": "shoe",
        "bags": "bag",
    }

    result = {}
    for _, row in df.iterrows():
        cid = canon_category_id(row[0])
        if cid is None or cid in result:
            continue

        fine_category = str(row[1]).strip().lower() if pd.notna(row[1]) else ""
        major_category = str(row[2]).strip().lower() if pd.notna(row[2]) else ""

        result[cid] = {
            "fine_category": fine_category,
            "major_category": major_category,
            "fastfit_category": major_to_fastfit.get(major_category, None),
        }
    return result

CATEGORY_INFO = load_category_major_map(CATEGORY_CSV)

# ------------------------------------------------------------
# 6) dataset + model
# ------------------------------------------------------------
normalize = torchvision.transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)
transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((img_size, img_size)),
    torchvision.transforms.ToTensor(),
    normalize,
])

base_dat = UIUCPolyvoreRetrievalDataset(
    datadir=str(POLYVORE_DATA_PARENT),
    txt_feat_path=str(TXT_FEAT_PATH),
    img_feat_path=str(IMG_FEAT_PATH),
    category_feat_path=str(CATEGORY_FEAT_PATH),
    outfit_feat_path=str(OUTFIT_FEAT_PATH),
    polyvore_split=POLYVORE_SPLIT,
    split="test",
    max_item_len=max_item_len,
    transform=transform,
    sample_hard_negative=False,
)

model = OutfitTransformerRetrieval(
    img_encoder=LinearImageEncoder(img_inp_size=img_inp_size, img_emb_size=img_emb_size),
    text_encoder=LinearTextEncoder(txt_inp_size=txt_inp_size, txt_emb_size=txt_emb_size),
    outfit_txt_encoder=LinearTextEncoder(
        txt_inp_size=outfit_txt_inp_size,
        txt_emb_size=txt_emb_size + img_emb_size,
    ) if use_outfit_txt else None,
    nhead=nhead,
    num_layers=num_layers,
    margin=margin,
    target_item_info="category",
    use_outfit_txt=use_outfit_txt,
)

checkpoint = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(checkpoint["model"], strict=True)
model = model.to(device)
model.eval()

print("checkpoint loaded:", CKPT_PATH)

# ------------------------------------------------------------
# 7) FashionCLIP text encoder
# ------------------------------------------------------------
from fashion_clip.fashion_clip import FashionCLIP
fclip = FashionCLIP("fashion-clip")

def encode_outfit_text(text: str, normalize=False):
    arr = fclip.encode_text([text], batch_size=1)[0].astype(np.float32)
    if normalize:
        norm = np.linalg.norm(arr)
        if norm > 0:
            arr = arr / norm
    return torch.tensor(arr, dtype=torch.float32, device=device)

# ------------------------------------------------------------
# 8) build item_meta
# ------------------------------------------------------------
def build_item_meta_from_fg(base_dat):
    item_meta = {}
    for item_id in base_dat.category_feats.keys():
        item_id = str(item_id)
        fg_id = canon_category_id(base_dat.im2fg.get(item_id, None))
        cat_info = CATEGORY_INFO.get(fg_id)

        if cat_info is None:
            fine_category = None
            major_category = None
            fastfit_category = None
        else:
            fine_category = cat_info["fine_category"]
            major_category = cat_info["major_category"]
            fastfit_category = cat_info["fastfit_category"]

        image_path = first_existing_image(item_id)
        item_meta[item_id] = {
            "item_id": item_id,
            "fg_id": fg_id,
            "fine_category": fine_category,
            "major_category": major_category,
            "fastfit_category": fastfit_category,
            "image_path": str(image_path) if image_path else None,
            "title": "",
        }
    return item_meta

item_meta = build_item_meta_from_fg(base_dat)

# ------------------------------------------------------------
# 9) candidate bank
# ------------------------------------------------------------
def extract_item_embedding(item_id: str):
    with torch.no_grad(), ctx:
        emb = base_dat.extract_emb({"item_id": str(item_id)}, model, device, ctx, to_numpy=False)
    return emb.detach().cpu().float()

def build_candidate_bank(item_meta, cache_path=None):
    if cache_path is not None and Path(cache_path).exists():
        print("load candidate bank from cache:", cache_path)
        return torch.load(cache_path, map_location="cpu")

    category_to_ids = defaultdict(list)
    for item_id, meta in item_meta.items():
        ff_cat = meta.get("fastfit_category")
        if ff_cat is None:
            continue
        if meta.get("image_path") is None:
            continue
        if str(item_id) not in base_dat.category_feats:
            continue
        category_to_ids[ff_cat].append(str(item_id))

    candidate_bank = {}
    for ff_cat, ids in category_to_ids.items():
        emb_list = []
        good_ids = []
        for item_id in tqdm(ids, desc=f"build bank: {ff_cat}"):
            try:
                emb = extract_item_embedding(item_id)
                emb_list.append(emb)
                good_ids.append(item_id)
            except Exception:
                pass

        if good_ids:
            candidate_bank[ff_cat] = {
                "ids": good_ids,
                "embs": torch.cat(emb_list, dim=0),
            }

    if cache_path is not None:
        torch.save(candidate_bank, cache_path)
    return candidate_bank

candidate_bank = build_candidate_bank(item_meta, BANK_CACHE)

def build_category_condition_bank(candidate_bank):
    cond_bank = {}
    for ff_cat, pack in candidate_bank.items():
        vecs = []
        for item_id in pack["ids"][:512]:
            if str(item_id) in base_dat.category_feats:
                vec = torch.tensor(base_dat.category_feats[str(item_id)], dtype=torch.float32)
                vecs.append(vec)
        if vecs:
            cond_bank[ff_cat] = torch.stack(vecs, dim=0).mean(dim=0).to(device)
    return cond_bank

category_condition_bank = build_category_condition_bank(candidate_bank)
print("category condition bank keys:", list(category_condition_bank.keys()))

# ------------------------------------------------------------
# 10) keep-k utilities
# ------------------------------------------------------------
def infer_empty_shapes():
    any_item_id = next(iter(base_dat.category_feats.keys()))
    imgs, txts, _ = base_dat.load_outfit({"items": [{"item_id": str(any_item_id)}]})
    padded_imgs, padded_txts, padded_mask = base_dat.pad_imgs_and_txts(imgs, txts)
    return padded_imgs.shape[-1], padded_txts.shape[-1], padded_mask.dtype

EMPTY_IMG_DIM, EMPTY_TXT_DIM, MASK_DTYPE = infer_empty_shapes()

def prepare_partial_inputs(partial_item_ids):
    if len(partial_item_ids) == 0:
        partial_imgs = torch.zeros((max_item_len, EMPTY_IMG_DIM), dtype=torch.float32, device=device)
        partial_txts = torch.zeros((max_item_len, EMPTY_TXT_DIM), dtype=torch.float32, device=device)
        mask = torch.ones((max_item_len,), dtype=torch.bool, device=device)
        return partial_imgs, partial_txts, mask

    items = [{"item_id": str(iid)} for iid in partial_item_ids]
    partial_imgs, partial_txts, _ = base_dat.load_outfit({"items": items})
    partial_imgs, partial_txts, mask = base_dat.pad_imgs_and_txts(partial_imgs, partial_txts)
    return partial_imgs.to(device), partial_txts.to(device), mask.to(device)

def rank_candidates_for_category(logits, ff_cat, exclude_ids=None, topk=20):
    exclude_ids = set(str(x) for x in (exclude_ids or []))
    pack = candidate_bank[ff_cat]
    sims = F.cosine_similarity(logits.cpu(), pack["embs"], dim=1)
    order = torch.argsort(sims, descending=True).tolist()

    ranked = []
    for idx in order:
        item_id = str(pack["ids"][idx])
        if item_id in exclude_ids:
            continue
        ranked.append({
            "item_id": item_id,
            "score": float(sims[idx].item()),
            "fastfit_category": ff_cat,
            "image_path": item_meta[item_id]["image_path"],
        })
        if len(ranked) >= topk:
            break
    return ranked

def choose_candidate(ranked_items, sampling_topk=None):
    if not ranked_items:
        raise ValueError("沒有可選 candidate")
    if sampling_topk is None or sampling_topk <= 1:
        return ranked_items[0]

    top_items = ranked_items[:sampling_topk]
    scores = np.array([max(x["score"], 1e-8) for x in top_items], dtype=np.float64)
    probs = scores / scores.sum()
    idx = np.random.choice(len(top_items), p=probs)
    return top_items[idx]

def extract_fastfit_item_ids_from_outfit(outfit):
    refs = {"upper": None, "lower": None, "overall": None, "shoe": None, "bag": None}
    for item in outfit.get("items", []):
        item_id = str(item.get("item_id"))
        meta = item_meta.get(item_id, {})
        ff_cat = meta.get("fastfit_category")
        if ff_cat in refs and refs[ff_cat] is None:
            refs[ff_cat] = item_id
    return refs

def infer_schema_from_item_ids(item_ids_by_cat):
    schema = [label for label in DISPLAY_ORDER if item_ids_by_cat.get(label) is not None]

    if "overall" in schema and ("upper" in schema or "lower" in schema):
        schema = [x for x in schema if x != "overall"]

    return schema

def refs_from_item_ids_by_cat(item_ids_by_cat):
    refs = {"upper": None, "lower": None, "overall": None, "shoe": None, "bag": None}
    for ff_cat, item_id in item_ids_by_cat.items():
        if item_id is None:
            continue
        meta = item_meta.get(str(item_id))
        if meta is not None:
            refs[ff_cat] = meta.get("image_path")
    return refs

def build_final_refs(kept_item_ids, generated_rows):
    refs = {"upper": None, "lower": None, "overall": None, "shoe": None, "bag": None}

    for item_id in kept_item_ids:
        item_id = str(item_id)
        meta = item_meta[item_id]
        ff_cat = meta["fastfit_category"]
        refs[ff_cat] = meta["image_path"]

    for row in generated_rows:
        refs[row["target_fastfit_category"]] = row["image_path"]

    return refs

def select_keep_categories(gt_schema, keep_n, keep_categories=None, keep_mode="prefix"):
    if keep_n == 0:
        return []

    if keep_categories is not None:
        filtered = [c for c in gt_schema if c in keep_categories]
        if len(filtered) != keep_n:
            raise ValueError(
                f"指定 keep_categories={keep_categories} 與 keep_n={keep_n} 不一致，"
                f"或有類別不在 gt_schema={gt_schema} 中"
            )
        return filtered

    if keep_mode == "prefix":
        return gt_schema[:keep_n]
    elif keep_mode == "suffix":
        return gt_schema[-keep_n:]
    else:
        raise ValueError("keep_mode 只支援 'prefix' 或 'suffix'")

def build_keep_setup(outfit, keep_n=0, keep_mode="prefix", keep_categories=None):
    gt_item_ids_by_cat = extract_fastfit_item_ids_from_outfit(outfit)
    gt_schema = infer_schema_from_item_ids(gt_item_ids_by_cat)

    if keep_n > len(gt_schema):
        raise ValueError(f"keep_n={keep_n} 大於可用 schema 長度 {len(gt_schema)}")

    kept_categories = select_keep_categories(
        gt_schema=gt_schema,
        keep_n=keep_n,
        keep_categories=keep_categories,
        keep_mode=keep_mode,
    )

    kept_item_ids = [str(gt_item_ids_by_cat[c]) for c in kept_categories]
    gen_schema = [c for c in gt_schema if c not in kept_categories]

    return {
        "gt_item_ids_by_cat": gt_item_ids_by_cat,
        "gt_schema": gt_schema,
        "keep_n": keep_n,
        "keep_categories": kept_categories,
        "kept_item_ids": kept_item_ids,
        "gen_schema": gen_schema,
    }

def generate_outfit_keepk(
    outfit_text,
    schema,
    kept_item_ids=None,
    topk_per_step=20,
    sampling_topk=None,
    verbose=True,
):
    kept_item_ids = [str(x) for x in (kept_item_ids or [])]
    outfit_txt = encode_outfit_text(outfit_text, normalize=NORMALIZE_OUTFIT_TEXT)

    partial_item_ids = kept_item_ids.copy()
    step_results = []

    for step_idx, ff_cat in enumerate(schema, start=1):
        if ff_cat not in category_condition_bank:
            raise KeyError(f"category_condition_bank 缺少類別: {ff_cat}")

        partial_imgs, partial_txts, mask = prepare_partial_inputs(partial_item_ids)
        positive_category = category_condition_bank[ff_cat]

        with torch.no_grad(), ctx:
            out = model(
                partial_imgs.unsqueeze(0),
                partial_txts.unsqueeze(0),
                mask.unsqueeze(0),
                positive_category=positive_category.unsqueeze(0),
                outfit_txt=outfit_txt.unsqueeze(0),
            )
            logits = out["logits"].detach().cpu().float()

        ranked = rank_candidates_for_category(
            logits=logits,
            ff_cat=ff_cat,
            exclude_ids=partial_item_ids,
            topk=topk_per_step,
        )
        chosen = choose_candidate(ranked, sampling_topk=sampling_topk)

        partial_item_ids.append(chosen["item_id"])

        step_results.append({
            "step": step_idx,
            "target_fastfit_category": ff_cat,
            "chosen_item_id": chosen["item_id"],
            "chosen_score": chosen["score"],
            "image_path": chosen["image_path"],
        })

        if verbose:
            print(f"[step {step_idx}] {ff_cat:>7s} -> {chosen['item_id']} | score={chosen['score']:.4f}")

    scores = [x["chosen_score"] for x in step_results]
    mean_score = float(np.mean(scores)) if len(scores) > 0 else None

    return {
        "outfit_text": outfit_text,
        "kept_item_ids": kept_item_ids,
        "schema": schema,
        "items": step_results,
        "mean_score": mean_score,
    }

def run_keep_case(
    outfit_text,
    outfit,
    keep_n=0,
    keep_mode="prefix",
    keep_categories=None,
    topk_per_step=20,
    sampling_topk=None,
    verbose=True,
):
    setup = build_keep_setup(
        outfit=outfit,
        keep_n=keep_n,
        keep_mode=keep_mode,
        keep_categories=keep_categories,
    )

    result = generate_outfit_keepk(
        outfit_text=outfit_text,
        schema=setup["gen_schema"],
        kept_item_ids=setup["kept_item_ids"],
        topk_per_step=topk_per_step,
        sampling_topk=sampling_topk,
        verbose=verbose,
    )

    final_refs = build_final_refs(setup["kept_item_ids"], result["items"])

    kept_map = {c: False for c in DISPLAY_ORDER}
    gen_map = {c: False for c in DISPLAY_ORDER}

    for c in setup["keep_categories"]:
        kept_map[c] = True
    for c in setup["gen_schema"]:
        gen_map[c] = True

    return {
        **setup,
        "result": result,
        "final_refs": final_refs,
        "kept_map": kept_map,
        "gen_map": gen_map,
    }

# ------------------------------------------------------------
# 11) load selected case
# ------------------------------------------------------------
with open(NEW_TITLE_JSON, "r", encoding="utf-8") as f:
    new_title_map_raw = json.load(f)

new_title_map = {}
for sid, value in new_title_map_raw.items():
    new_title_map[str(sid)] = normalize_text_value(value)

assert CASE_SET_ID in new_title_map, f"new_polyvore_outfit_titles.json 找不到 set_id={CASE_SET_ID}"
selected_text = new_title_map[CASE_SET_ID]

with open(TEST_JSON, "r", encoding="utf-8") as f:
    test_outfits = json.load(f)

test_outfit_map = {str(x["set_id"]): x for x in test_outfits}
assert CASE_SET_ID in test_outfit_map, f"test.json 找不到 set_id={CASE_SET_ID}"
selected_outfit = test_outfit_map[CASE_SET_ID]

print("Selected set_id =", CASE_SET_ID)
print("Query text =", selected_text)

# ------------------------------------------------------------
# 12) GT
# ------------------------------------------------------------
gt_item_ids_by_cat = extract_fastfit_item_ids_from_outfit(selected_outfit)
gt_schema = infer_schema_from_item_ids(gt_item_ids_by_cat)
gt_refs = refs_from_item_ids_by_cat(gt_item_ids_by_cat)

print("GT item ids by cat =", gt_item_ids_by_cat)
print("GT schema =", gt_schema)

if len(gt_schema) < 2:
    raise ValueError(
        f"目前這個 case 的 GT schema={gt_schema} 少於 2 個可用類別，無法同時展示 keep0/1/2。"
    )

# ------------------------------------------------------------
# 13) run keep0 / keep1 / keep2
# ------------------------------------------------------------
keep0_case = run_keep_case(
    outfit_text=selected_text,
    outfit=selected_outfit,
    keep_n=0,
    keep_mode=KEEP_MODE_IF_NOT_SPECIFIED,
    keep_categories=None,
    topk_per_step=TOPK_PER_STEP,
    sampling_topk=SAMPLING_TOPK,
    verbose=True,
)

keep1_case = run_keep_case(
    outfit_text=selected_text,
    outfit=selected_outfit,
    keep_n=1,
    keep_mode=KEEP_MODE_IF_NOT_SPECIFIED,
    keep_categories=KEEP1_CATEGORIES,
    topk_per_step=TOPK_PER_STEP,
    sampling_topk=SAMPLING_TOPK,
    verbose=True,
)

keep2_case = run_keep_case(
    outfit_text=selected_text,
    outfit=selected_outfit,
    keep_n=2,
    keep_mode=KEEP_MODE_IF_NOT_SPECIFIED,
    keep_categories=KEEP2_CATEGORIES,
    topk_per_step=TOPK_PER_STEP,
    sampling_topk=SAMPLING_TOPK,
    verbose=True,
)

print("\n========== SUMMARY ==========")
print("GT schema        :", gt_schema)
print("Keep0 kept cats  :", keep0_case["keep_categories"])
print("Keep0 gen schema :", keep0_case["gen_schema"])
print("Keep0 mean score :", keep0_case["result"]["mean_score"])

print("Keep1 kept cats  :", keep1_case["keep_categories"])
print("Keep1 gen schema :", keep1_case["gen_schema"])
print("Keep1 mean score :", keep1_case["result"]["mean_score"])

print("Keep2 kept cats  :", keep2_case["keep_categories"])
print("Keep2 gen schema :", keep2_case["gen_schema"])
print("Keep2 mean score :", keep2_case["result"]["mean_score"])

# ------------------------------------------------------------
# 14) save JSON summary
# ------------------------------------------------------------
if SAVE_JSON:
    save_pack = {
        "case_set_id": CASE_SET_ID,
        "query_text": selected_text,
        "gt_schema": gt_schema,
        "gt_item_ids_by_cat": gt_item_ids_by_cat,
        "keep0": {
            "keep_categories": keep0_case["keep_categories"],
            "gen_schema": keep0_case["gen_schema"],
            "result": keep0_case["result"],
            "final_refs": keep0_case["final_refs"],
        },
        "keep1": {
            "keep_categories": keep1_case["keep_categories"],
            "gen_schema": keep1_case["gen_schema"],
            "result": keep1_case["result"],
            "final_refs": keep1_case["final_refs"],
        },
        "keep2": {
            "keep_categories": keep2_case["keep_categories"],
            "gen_schema": keep2_case["gen_schema"],
            "result": keep2_case["result"],
            "final_refs": keep2_case["final_refs"],
        },
    }
    json_path = OUT_DIR / f"case_{CASE_SET_ID}_keep012_summary.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(save_pack, f, ensure_ascii=False, indent=2)
    print("saved json:", json_path)

# ------------------------------------------------------------
# 15) paper-style comparison figure
# ------------------------------------------------------------
def draw_border(ax, style="solid", linewidth=2.0, color="black"):
    rect = patches.Rectangle(
        (0, 0), 1, 1,
        transform=ax.transAxes,
        fill=False,
        linestyle=style,
        linewidth=linewidth,
        edgecolor=color
    )
    ax.add_patch(rect)

def plot_paper_style_grid(row_defs, save_path=None, suptitle=None):
    nrows = len(row_defs)
    ncols = len(DISPLAY_ORDER)

    fig, axes = plt.subplots(nrows, ncols, figsize=(2.2 * ncols, 2.6 * nrows))
    if nrows == 1:
        axes = np.expand_dims(axes, axis=0)

    for r, row in enumerate(row_defs):
        row_name = row["name"]
        refs = row["refs"]
        kept_map = row.get("kept_map", {c: False for c in DISPLAY_ORDER})
        gen_map = row.get("gen_map", {c: False for c in DISPLAY_ORDER})

        for c, ff_cat in enumerate(DISPLAY_ORDER):
            ax = axes[r, c]
            img_path = refs.get(ff_cat)
            img = open_image_or_none(img_path)

            if img is not None:
                ax.imshow(img)
            else:
                ax.set_facecolor("#eeeeee")
                ax.text(0.5, 0.5, "N/A", ha="center", va="center", fontsize=10, transform=ax.transAxes)

            ax.set_xticks([])
            ax.set_yticks([])

            # 標題列
            if r == 0:
                ax.set_title(ff_cat, fontsize=10, pad=8)

            # row label
            if c == 0:
                ax.text(
                    -0.25, 0.5, row_name,
                    transform=ax.transAxes,
                    ha="right", va="center",
                    fontsize=12, fontweight="bold"
                )

            # border style
            if row_name == "Original":
                draw_border(ax, style="solid", linewidth=1.6, color="black")
            else:
                if kept_map.get(ff_cat, False):
                    draw_border(ax, style="solid", linewidth=2.2, color="black")
                    ax.text(0.02, 0.02, "keep", transform=ax.transAxes,
                            ha="left", va="bottom", fontsize=8,
                            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="black", lw=0.8))
                elif gen_map.get(ff_cat, False):
                    draw_border(ax, style="dashed", linewidth=2.2, color="black")
                    ax.text(0.02, 0.02, "gen", transform=ax.transAxes,
                            ha="left", va="bottom", fontsize=8,
                            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="black", lw=0.8))
                else:
                    draw_border(ax, style="solid", linewidth=1.0, color="#BBBBBB")

    if suptitle is None:
        suptitle = f"Case {CASE_SET_ID} | Original / Keep2 / Keep1 / Keep0"

    fig.suptitle(suptitle, fontsize=14, y=0.995)
    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
        print("saved figure:", save_path)

    plt.show()

paper_rows = [
    {
        "name": "Original",
        "refs": gt_refs,
        "kept_map": {c: False for c in DISPLAY_ORDER},
        "gen_map": {c: False for c in DISPLAY_ORDER},
    },
    {
        "name": "Keep2",
        "refs": keep2_case["final_refs"],
        "kept_map": keep2_case["kept_map"],
        "gen_map": keep2_case["gen_map"],
    },
    {
        "name": "Keep1",
        "refs": keep1_case["final_refs"],
        "kept_map": keep1_case["kept_map"],
        "gen_map": keep1_case["gen_map"],
    },
    {
        "name": "Keep0",
        "refs": keep0_case["final_refs"],
        "kept_map": keep0_case["kept_map"],
        "gen_map": keep0_case["gen_map"],
    },
]

paper_fig_path = OUT_DIR / f"paper_style_case_{CASE_SET_ID}_keep012.png"
plot_paper_style_grid(
    row_defs=paper_rows,
    save_path=paper_fig_path,
    suptitle=f"Case {CASE_SET_ID} | Original / Keep2 / Keep1 / Keep0\n{selected_text}"
)

# ------------------------------------------------------------
# 16) show simple rows too
# ------------------------------------------------------------
show_ref_row(gt_refs, f"Original | set_id={CASE_SET_ID}")
show_ref_row(keep2_case["final_refs"], f"Keep2 | kept={keep2_case['keep_categories']}")
show_ref_row(keep1_case["final_refs"], f"Keep1 | kept={keep1_case['keep_categories']}")
show_ref_row(keep0_case["final_refs"], f"Keep0 | generated from empty")

# ------------------------------------------------------------
# 17) FastFit helper
# ------------------------------------------------------------
def run_fastfit_from_refs(target_refs, out_name_suffix="keep_case"):
    if not RUN_FASTFIT:
        print("RUN_FASTFIT=False，目前只顯示商品列。")
        return None, None, "RUN_FASTFIT=False"

    if not FASTFIT_REPO.exists():
        msg = f"[SKIP] 找不到 FastFit repo: {FASTFIT_REPO}"
        print(msg)
        return None, None, msg

    if str(FASTFIT_REPO) not in sys.path:
        sys.path.insert(0, str(FASTFIT_REPO))

    try:
        from app import FastFitDemo

        demo = FastFitDemo(
            base_model_path=str(FASTFIT_REPO / "Models" / "FastFit-MR-1024"),
            util_model_path=str(FASTFIT_REPO / "Models" / "Human-Toolkit"),
            mixed_precision="bf16",
            device="cuda" if torch.cuda.is_available() else "cpu",
        )

        person_img = Image.open(PERSON_IMAGE).convert("RGB")
        upper_img = Image.open(target_refs["upper"]).convert("RGB") if target_refs["upper"] else None
        lower_img = Image.open(target_refs["lower"]).convert("RGB") if target_refs["lower"] else None
        dress_img = Image.open(target_refs["overall"]).convert("RGB") if target_refs["overall"] else None
        shoe_img  = Image.open(target_refs["shoe"]).convert("RGB") if target_refs["shoe"] else None
        bag_img   = Image.open(target_refs["bag"]).convert("RGB") if target_refs["bag"] else None

        result_img, status = demo.generate_image(
            person_img=person_img,
            upper_img=upper_img,
            lower_img=lower_img,
            dress_img=dress_img,
            shoe_img=shoe_img,
            bag_img=bag_img,
            ref_height=768,
            num_inference_steps=30,
            guidance_scale=2.5,
            use_square_mask=False,
            seed=42,
            enable_pose=True,
        )

        print(f"FastFit status [{out_name_suffix}]:", status)

        if result_img is not None:
            out_path = OUT_DIR / f"fastfit_case_{CASE_SET_ID}_{out_name_suffix}.png"
            result_img.save(out_path)
            print("saved:", out_path)
            return result_img, out_path, status
        else:
            print("FastFit 沒有產生有效結果")
            return None, None, status

    except Exception as e:
        msg = f"[FastFit skipped] {repr(e)}"
        print(msg)
        return None, None, msg

def plot_tryon_triptych(results, save_path=None, title=None):
    labels = list(results.keys())
    n = len(labels)

    fig, axes = plt.subplots(1, n, figsize=(4.3 * n, 6.4))
    if n == 1:
        axes = [axes]

    for ax, label in zip(axes, labels):
        img = results[label]
        if img is not None:
            ax.imshow(img)
        else:
            ax.set_facecolor("#eeeeee")
            ax.text(0.5, 0.5, "No Result", ha="center", va="center", fontsize=12, transform=ax.transAxes)
        ax.set_title(label, fontsize=12)
        ax.axis("off")

    if title is None:
        title = f"FastFit Try-On | Keep2 / Keep1 / Keep0 | case={CASE_SET_ID}"
    fig.suptitle(title, fontsize=14, y=0.98)
    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
        print("saved figure:", save_path)

    plt.show()

# ------------------------------------------------------------
# 18) run FastFit for keep2 / keep1 / keep0
# ------------------------------------------------------------
keep2_tryon_img, keep2_tryon_path, keep2_status = run_fastfit_from_refs(
    keep2_case["final_refs"], out_name_suffix="keep2"
)
keep1_tryon_img, keep1_tryon_path, keep1_status = run_fastfit_from_refs(
    keep1_case["final_refs"], out_name_suffix="keep1"
)
keep0_tryon_img, keep0_tryon_path, keep0_status = run_fastfit_from_refs(
    keep0_case["final_refs"], out_name_suffix="keep0"
)

tryon_fig_path = OUT_DIR / f"fastfit_triptych_case_{CASE_SET_ID}_keep012.png"
plot_tryon_triptych(
    results={
        "Keep2": keep2_tryon_img,
        "Keep1": keep1_tryon_img,
        "Keep0": keep0_tryon_img,
    },
    save_path=tryon_fig_path,
    title=f"FastFit Try-On | Keep2 / Keep1 / Keep0\n{selected_text}"
)

# ------------------------------------------------------------
# 19) final print
# ------------------------------------------------------------
print("\n========== FILES ==========")
print("paper-style items figure :", paper_fig_path)
print("keep2 try-on image       :", keep2_tryon_path)
print("keep1 try-on image       :", keep1_tryon_path)
print("keep0 try-on image       :", keep0_tryon_path)
if SAVE_JSON:
    print("summary json             :", json_path)

print("\nDone.")

PROJECT_ROOT = /home/ester/valid_description/text-conditioned-outfit-recommendation
CASE_SET_ID = 224188768
OUT_DIR = /home/ester/valid_description/text-conditioned-outfit-recommendation/jupyter_outputs/case_224188768_keep012_full
FASTFIT_REPO exists = True


2026-04-02 02:19:54.501778: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-02 02:19:54.708672: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775067594.782847    6122 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775067594.807199    6122 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775067595.941036    6122 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

checkpoint loaded: /home/ester/valid_description/text-conditioned-outfit-recommendation/experiments/cir_new_seed1/ckpt.pt


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


load candidate bank from cache: /home/ester/valid_description/text-conditioned-outfit-recommendation/jupyter_outputs/candidate_bank_new_seed1_fg_based_v3.pt
category condition bank keys: ['lower', 'upper', 'shoe', 'bag', 'overall']
Selected set_id = 224188768
Query text = 26.2° C, Effortless Summer Wedding Style with a Pop of Citrus
GT item ids by cat = {'upper': None, 'lower': None, 'overall': '209144226', 'shoe': '208337886', 'bag': '201548065'}
GT schema = ['overall', 'shoe', 'bag']


...

FastFit status [keep1]: success_generation
saved: /home/ester/valid_description/text-conditioned-outfit-recommendation/jupyter_outputs/case_224188768_keep012_full/fastfit_case_224188768_keep1.png


An error occurred while trying to fetch /home/ester/valid_description/FastFit/Models/FastFit-MR-1024: Error no file named diffusion_pytorch_model.safetensors found in directory /home/ester/valid_description/FastFit/Models/FastFit-MR-1024.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
100%|███████████████████████████████████████████████████████████████████████████████████| 30/30 [00:02<00:00, 12.35it/s]


FastFit status [keep0]: success_generation
saved: /home/ester/valid_description/text-conditioned-outfit-recommendation/jupyter_outputs/case_224188768_keep012_full/fastfit_case_224188768_keep0.png
saved figure: /home/ester/valid_description/text-conditioned-outfit-recommendation/jupyter_outputs/case_224188768_keep012_full/fastfit_triptych_case_224188768_keep012.png

========== FILES ==========
paper-style items figure : /home/ester/valid_description/text-conditioned-outfit-recommendation/jupyter_outputs/case_224188768_keep012_full/paper_style_case_224188768_keep012.png
keep2 try-on image       : /home/ester/valid_description/text-conditioned-outfit-recommendation/jupyter_outputs/case_224188768_keep012_full/fastfit_case_224188768_keep2.png
keep1 try-on image       : /home/ester/valid_description/text-conditioned-outfit-recommendation/jupyter_outputs/case_224188768_keep012_full/fastfit_case_224188768_keep1.png
keep0 try-on image       : /home/ester/valid_description/text-conditioned-outfi

/tmp/ipykernel_6122/710511180.py:1032: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<Figure size 1806x896 with 3 Axes>

In [4]:
import os
import re
import json
import ast
import glob

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ──────────────────────────────────────────────────────────
# 0) 路徑設定（保留原本輸出位置）
# ──────────────────────────────────────────────────────────
EXP_ROOT           = "./experiments_none_only"
DETAIL_DIR         = os.path.join(EXP_ROOT, "detail_cir")
DETAIL_RUN_TAG     = "none_subset"
DETAIL_SUBSET_TAG  = "subset"

FRAG_JSONL          = "/home/ester/valid_description/wos_split_results_v5_merged_retry_round3.jsonl"
ORIGINAL_TITLE_JSON = "/home/ester/valid_description/polyvore_data/polyvore_outfits/polyvore_outfit_titles.json"
CATEGORY_CSV        = "/home/ester/valid_description/polyvore_data/polyvore_outfits/categories.csv"

OUTPUT_DIR = "./subset_analysis_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

VARIANTS = {
    "original"    : "outfitUrlTitle",
    "full"        : "NewoutfitUrlTitle",
    "no_weather"  : "no_weather",
    "no_occasion" : "no_occasion",
    "no_style"    : "no_style",
}


# ──────────────────────────────────────────────────────────
# 1) 工具函式
# ──────────────────────────────────────────────────────────
def normalize_text(x):
    if x is None:
        return ""
    if isinstance(x, str):
        return x.strip()
    if isinstance(x, list):
        return " | ".join(str(i).strip() for i in x if str(i).strip())
    return str(x).strip()

def normalize_token_text(x):
    if x is None:
        return ""
    x = str(x).strip().lower()
    x = re.sub(r"[_/|,-]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def parse_top_ids(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [str(i) for i in x]
    try:
        v = json.loads(x)
        if isinstance(v, list):
            return [str(i) for i in v]
    except Exception:
        pass
    try:
        v = ast.literal_eval(x)
        if isinstance(v, list):
            return [str(i) for i in v]
    except Exception:
        pass
    return []

def ordered_unique(seq):
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


# ──────────────────────────────────────────────────────────
# 2) 讀取資料
# ──────────────────────────────────────────────────────────
def load_one_variant_detail(variant_name):
    pattern = os.path.join(
        DETAIL_DIR,
        f"detail_cir_{variant_name}_{DETAIL_SUBSET_TAG}_{DETAIL_RUN_TAG}_seed*.csv"
    )
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"找不到 detail 檔案：{pattern}")

    dfs = []
    for fp in files:
        df = pd.read_csv(fp)
        df["variant"] = variant_name
        df["source_file"] = os.path.basename(fp)
        dfs.append(df)

    out = pd.concat(dfs, ignore_index=True)

    required = ["seed", "set_id", "target_item_id", "target_item_fg", "rank", "hit@10", "top10_ids"]
    for c in required:
        if c not in out.columns:
            raise KeyError(f"detail csv 缺少必要欄位：{c}")

    out["seed"]           = out["seed"].astype(int)
    out["set_id"]         = out["set_id"].astype(str)
    out["target_item_id"] = out["target_item_id"].astype(str)
    out["target_item_fg"] = out["target_item_fg"].astype(str)
    out["rank"]           = pd.to_numeric(out["rank"], errors="coerce")
    out["hit@10"]         = pd.to_numeric(out["hit@10"], errors="coerce")
    out["top10_ids_list"] = out["top10_ids"].apply(parse_top_ids)
    out["query_key"]      = (
        out["seed"].astype(str) + "||" + out["set_id"] + "||" + out["target_item_id"]
    )
    return out

def load_original_titles(path):
    if not os.path.exists(path):
        print(f"[WARN] 找不到 {path}，original text 將為空字串")
        return pd.DataFrame(columns=["set_id", "original_text"])

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []
    for sid, value in data.items():
        if isinstance(value, str):
            text = value.strip()
        elif isinstance(value, dict):
            url   = normalize_text(value.get("url_name", value.get("url", "")))
            title = normalize_text(value.get("title", ""))
            text  = " ".join(p for p in [url, title] if p).strip()
        else:
            text = normalize_text(value)

        rows.append({
            "set_id": str(sid),
            "original_text": text
        })

    return pd.DataFrame(rows)

def load_fragments(path):
    if not os.path.exists(path):
        print(f"[WARN] 找不到 {path}，fragment 欄位將為空")
        return pd.DataFrame(columns=[
            "set_id", "title_full_text",
            "weather_terms", "occasion_terms", "style_terms",
            "weather_text", "occasion_text", "style_text",
            "title_no_weather_text", "title_no_occasion_text", "title_no_style_text"
        ])

    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            obj = json.loads(line)
            if obj.get("valid") is not True:
                continue

            frags = obj.get("fragments", {})
            title_ablation = obj.get("title_ablation", {})

            w = frags.get("weather", [])
            o = frags.get("occasion", [])
            s = frags.get("style", [])

            rows.append({
                "set_id": str(obj.get("id", "")),
                "title_full_text": normalize_text(obj.get("title", "")),
                "weather_terms": w if isinstance(w, list) else [],
                "occasion_terms": o if isinstance(o, list) else [],
                "style_terms": s if isinstance(s, list) else [],
                "weather_text": normalize_text(w),
                "occasion_text": normalize_text(o),
                "style_text": normalize_text(s),
                "title_no_weather_text": normalize_text(title_ablation.get("no_weather", "")),
                "title_no_occasion_text": normalize_text(title_ablation.get("no_occasion", "")),
                "title_no_style_text": normalize_text(title_ablation.get("no_style", "")),
            })

    return pd.DataFrame(rows)

def load_category_map(csv_path):
    """
    預期 categories.csv:
      col0 = category_id
      col1 = fine_category
      col2 = major_category
    """
    empty = pd.DataFrame(columns=["category_id", "fine_category", "major_category"])

    if not os.path.exists(csv_path):
        print(f"[WARN] 找不到 {csv_path}")
        return empty

    try:
        df_raw = pd.read_csv(csv_path, header=None, dtype=str).fillna("")
    except Exception as e:
        print(f"[WARN] 讀取 categories.csv 失敗：{e}")
        return empty

    if df_raw.empty:
        return empty

    out = pd.DataFrame()
    out["category_id"] = df_raw.iloc[:, 0].astype(str).str.strip()
    out["fine_category"] = df_raw.iloc[:, 1].astype(str).str.strip() if df_raw.shape[1] >= 2 else ""
    out["major_category"] = df_raw.iloc[:, 2].astype(str).str.strip() if df_raw.shape[1] >= 3 else ""

    out["category_id"] = out["category_id"].astype(str)
    out["fine_category"] = out["fine_category"].apply(normalize_token_text)
    out["major_category"] = out["major_category"].apply(normalize_token_text)

    out = out.drop_duplicates(subset=["category_id"], keep="first").reset_index(drop=True)
    return out


# ──────────────────────────────────────────────────────────
# 3) 類別資訊對應（只用 categories.csv）
# ──────────────────────────────────────────────────────────
def attach_category_metadata(combined, category_df):
    combined = combined.copy()

    if category_df.empty:
        combined["fine_category"] = ""
        combined["major_category"] = ""
        return combined

    combined = combined.merge(
        category_df[["category_id", "fine_category", "major_category"]],
        left_on="target_item_fg",
        right_on="category_id",
        how="left"
    )

    combined["fine_category"] = combined["fine_category"].fillna("")
    combined["major_category"] = combined["major_category"].fillna("")

    matched = (combined["fine_category"].str.len() > 0) | (combined["major_category"].str.len() > 0)
    print(f"  category metadata matched: {matched.sum()} / {len(combined)} rows ({matched.mean():.2%})")

    return combined


# ──────────────────────────────────────────────────────────
# 4) 溫度抽取
# ──────────────────────────────────────────────────────────
TEMP_PATTERN = re.compile(r"(-?\d+(?:\.\d+)?)\s*°?\s*([CFcf])")

def extract_temperature_celsius_from_text(text):
    """
    支援：
      24°C
      24 C
      75°F
    若為 F，轉成 C
    """
    if not isinstance(text, str) or not text.strip():
        return np.nan

    m = TEMP_PATTERN.search(text)
    if not m:
        return np.nan

    value = float(m.group(1))
    unit = m.group(2).upper()

    if unit == "F":
        value = (value - 32.0) * 5.0 / 9.0
    return value

def extract_row_temperature_c(row):
    candidates = [
        row.get("weather_text", ""),
        row.get("title_full_text", ""),
        row.get("original_text", ""),
    ]
    for txt in candidates:
        temp = extract_temperature_celsius_from_text(txt)
        if pd.notna(temp):
            return temp
    return np.nan

def add_temperature_column(combined):
    combined = combined.copy()
    combined["temperature_c"] = combined.apply(extract_row_temperature_c, axis=1)

    valid = combined["temperature_c"].dropna()
    if len(valid) == 0:
        threshold = np.nan
        print("[WARN] 沒有抓到任何溫度，冷熱子集將為空")
    else:
        threshold = float(valid.median())
        print(f"  extracted temperature rows: {len(valid)} / {len(combined)}")
        print(f"  temperature median threshold: {threshold:.2f}°C")

    return combined, threshold


# ──────────────────────────────────────────────────────────
# 5) 子集定義
# ──────────────────────────────────────────────────────────
def derive_category_groups(category_df):
    """
    依 categories.csv 真實 major_category 值來做分組

    衣物主導:
        all body, bottoms, tops, outerwear

    配件主導:
        bags, shoes, accessories, hats, jewellery, scarves, sunglasses
    """
    all_major = ordered_unique(
        [normalize_token_text(x) for x in category_df["major_category"].astype(str).tolist() if str(x).strip()]
    )

    clothing_candidates = {"all body", "bottoms", "tops", "outerwear"}
    accessory_candidates = {"bags", "shoes", "accessories", "hats", "jewellery", "scarves", "sunglasses"}

    clothing_major_cats = [x for x in all_major if x in clothing_candidates]
    accessory_major_cats = [x for x in all_major if x in accessory_candidates]

    return all_major, clothing_major_cats, accessory_major_cats

def build_subset_dimensions(temp_threshold, clothing_major_cats, accessory_major_cats):
    # --- 1) Cold vs Warm ---
    def is_cold(row):
        t = row.get("temperature_c", np.nan)
        return pd.notna(t) and pd.notna(temp_threshold) and t <= temp_threshold

    def is_warm(row):
        t = row.get("temperature_c", np.nan)
        return pd.notna(t) and pd.notna(temp_threshold) and t > temp_threshold

    # --- 2) Formal vs Casual ---
    FORMAL_KEYWORDS = r"\b(formal|office|business|work|professional|interview|meeting|wedding|party|cocktail|elegant|tailored|blazer|suit)\b"
    CASUAL_KEYWORDS = r"\b(casual|weekend|streetwear|street|relaxed|everyday|daily|chill|laid-back|lounge|vacation|travel)\b"

    def occasion_source_text(row):
        return " ".join([
            normalize_text(row.get("occasion_text", "")),
            normalize_text(row.get("title_full_text", "")),
        ]).lower()

    def is_formal(row):
        return bool(re.search(FORMAL_KEYWORDS, occasion_source_text(row)))

    def is_casual(row):
        return bool(re.search(CASUAL_KEYWORDS, occasion_source_text(row)))

    # --- 3) Style richness ---
    def style_richness(row):
        terms = row.get("style_terms", [])
        if not isinstance(terms, list):
            return 0
        return len([t for t in terms if str(t).strip()])

    def is_high_style(row):
        return style_richness(row) >= 2

    def is_low_style(row):
        return style_richness(row) <= 1

    # --- 4) Clothing-led vs Accessory-led ---
    clothing_major_cats = set(clothing_major_cats)
    accessory_major_cats = set(accessory_major_cats)

    def is_clothing_led(row):
        return normalize_token_text(row.get("major_category", "")) in clothing_major_cats

    def is_accessory_led(row):
        return normalize_token_text(row.get("major_category", "")) in accessory_major_cats

    subset_dimensions = {
        "Weather: Cold vs Warm": [
            ("Cold", is_cold),
            ("Warm", is_warm),
        ],
        "Occasion: Formal vs Casual": [
            ("Formal", is_formal),
            ("Casual", is_casual),
        ],
        "Style Richness: High vs Low": [
            ("High style (≥2 terms)", is_high_style),
            ("Low style (≤1 term)", is_low_style),
        ],
        "Category: Clothing vs Accessory": [
            ("Clothing-led", is_clothing_led),
            ("Accessory-led", is_accessory_led),
        ],
    }

    return subset_dimensions


# ──────────────────────────────────────────────────────────
# 6) 建立 combined dataframe
# ──────────────────────────────────────────────────────────
def build_combined(variants):
    print("Loading detail CSVs ...")
    details = {}
    for short, vname in variants.items():
        details[short] = load_one_variant_detail(vname)
        print(f"  {short:12s}: {len(details[short])} rows")

    orig_df = load_original_titles(ORIGINAL_TITLE_JSON)
    frag_df = load_fragments(FRAG_JSONL)
    cat_df  = load_category_map(CATEGORY_CSV)

    print(f"  original titles: {len(orig_df)}")
    print(f"  fragments:       {len(frag_df)}")
    print(f"  category map:    {len(cat_df)}")

    base_cols = [
        "query_key", "seed", "set_id", "target_item_id", "target_item_fg",
        "rank", "hit@10", "top10_ids_list"
    ]

    left = details["original"][base_cols].rename(
        columns={
            "rank": "rank_orig",
            "hit@10": "hit10_orig",
            "top10_ids_list": "top10_orig"
        }
    )

    right = details["full"][["query_key", "rank", "hit@10", "top10_ids_list"]].rename(
        columns={
            "rank": "rank_full",
            "hit@10": "hit10_full",
            "top10_ids_list": "top10_full"
        }
    )

    combined = (
        left.merge(right, on="query_key", how="inner")
            .merge(orig_df, on="set_id", how="left")
            .merge(frag_df, on="set_id", how="left")
    )

    for short in ["no_weather", "no_occasion", "no_style"]:
        slim = details[short][["query_key", "rank", "hit@10"]].rename(
            columns={
                "rank": f"rank_{short}",
                "hit@10": f"hit10_{short}"
            }
        )
        combined = combined.merge(slim, on="query_key", how="left")

    combined = attach_category_metadata(combined, cat_df)
    combined, temp_threshold = add_temperature_column(combined)

    print(f"Combined rows: {len(combined)}")
    return combined, cat_df, temp_threshold


# ──────────────────────────────────────────────────────────
# 7) 子集評估
# ──────────────────────────────────────────────────────────
METRIC_PAIRS = [
    ("hit10_orig",        "hit10_full", "Hit@10"),
    ("rank_orig",         "rank_full",  "Median Rank"),
    ("hit10_no_weather",  "hit10_full", "Hit@10 (no_weather→full)"),
    ("hit10_no_occasion", "hit10_full", "Hit@10 (no_occasion→full)"),
    ("hit10_no_style",    "hit10_full", "Hit@10 (no_style→full)"),
]

def evaluate_subset(df_sub, label):
    result = {"n": int(len(df_sub)), "label": label}

    if len(df_sub) == 0:
        return result

    for left_col, right_col, name in METRIC_PAIRS:
        if left_col not in df_sub.columns or right_col not in df_sub.columns:
            continue

        left_vals = pd.to_numeric(df_sub[left_col], errors="coerce").dropna()
        right_vals = pd.to_numeric(df_sub[right_col], errors="coerce").dropna()

        if "Rank" in name:
            left_v = float(left_vals.median()) if len(left_vals) else float("nan")
            right_v = float(right_vals.median()) if len(right_vals) else float("nan")
        else:
            left_v = float(left_vals.mean()) if len(left_vals) else float("nan")
            right_v = float(right_vals.mean()) if len(right_vals) else float("nan")

        result[f"{name}_left"] = left_v
        result[f"{name}_right"] = right_v
        result[f"Δ{name}"] = right_v - left_v

    return result

def run_subset_analysis(combined, subset_dimensions):
    all_results = []

    for dim_name, subsets in subset_dimensions.items():
        print(f"\n  Dimension: {dim_name}")
        dim_results = []

        for subset_label, filter_fn in subsets:
            mask = combined.apply(filter_fn, axis=1)
            df_sub = combined[mask]
            res = evaluate_subset(df_sub, subset_label)
            res["dimension"] = dim_name
            dim_results.append(res)
            print(f"    [{subset_label}] n={res['n']}")

        mask_any = combined.apply(lambda row: any(fn(row) for _, fn in subsets), axis=1)
        res_all = evaluate_subset(combined[mask_any], "All (union)")
        res_all["dimension"] = dim_name
        dim_results.append(res_all)

        all_results.extend(dim_results)

    res_global = evaluate_subset(combined, "Full dataset")
    res_global["dimension"] = "Overall"
    all_results.append(res_global)

    return all_results

def sort_result_rows(rows, subset_dimensions):
    dimension_order = list(subset_dimensions.keys())
    subset_order = {
        dim: [label for label, _ in subsets]
        for dim, subsets in subset_dimensions.items()
    }

    def _key(r):
        dim = r.get("dimension", "")
        label = r.get("label", "")
        dim_idx = dimension_order.index(dim) if dim in dimension_order else 999
        label_list = subset_order.get(dim, [])
        label_idx = label_list.index(label) if label in label_list else 999
        return (dim_idx, label_idx, label)

    return sorted(rows, key=_key)


# ──────────────────────────────────────────────────────────
# 8) 視覺化
# ──────────────────────────────────────────────────────────
def plot_hitrate_comparison(all_results, save_dir, subset_dimensions):
    metric_key = "Hit@10"
    left_key = f"{metric_key}_left"
    right_key = f"{metric_key}_right"

    rows = [
        r for r in all_results
        if left_key in r and right_key in r
        and r.get("n", 0) > 0
        and r.get("label") != "All (union)"
        and r.get("dimension") != "Overall"
    ]
    rows = sort_result_rows(rows, subset_dimensions)

    if not rows:
        print("[WARN] plot_hitrate_comparison: no data")
        return

    labels = [f"{r['label']}\n(n={r['n']})" for r in rows]
    orig_vals = [r[left_key] for r in rows]
    prop_vals = [r[right_key] for r in rows]

    x = np.arange(len(labels))
    width = 0.36

    fig, ax = plt.subplots(figsize=(max(8, len(labels) * 1.6), 5.5))
    bars1 = ax.bar(x - width / 2, orig_vals, width, label="Original (Baseline)", color="#888888", edgecolor="white")
    bars2 = ax.bar(x + width / 2, prop_vals, width, label="Full (Proposed)", color="#2166AC", edgecolor="white")

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=7.5)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=7.5, color="#2166AC")

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8.5)
    ax.set_ylabel("Hit@10", fontsize=11)
    ax.set_title("Hit@10: Original vs Proposed Across Subsets", fontsize=13, fontweight="bold", pad=12)
    ax.legend(fontsize=10)
    ax.set_ylim(0, max(max(orig_vals), max(prop_vals), 0.01) * 1.18)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    prev_dim = None
    for i, r in enumerate(rows):
        if r["dimension"] != prev_dim and i > 0:
            ax.axvline(i - 0.5, color="#cccccc", linewidth=1, linestyle="--")
        prev_dim = r["dimension"]

    plt.tight_layout()
    path = os.path.join(save_dir, "fig1_hitrate_comparison.png")
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    print(f"[saved] {path}")

def plot_delta_heatmap(all_results, save_dir, subset_dimensions):
    metric_key = "Hit@10"
    delta_key = f"Δ{metric_key}"

    rows = [
        r for r in all_results
        if delta_key in r
        and r.get("label") not in ("All (union)", "Full dataset")
        and r.get("n", 0) > 0
        and r.get("dimension") != "Overall"
    ]
    rows = sort_result_rows(rows, subset_dimensions)

    if not rows:
        print("[WARN] plot_delta_heatmap: no data")
        return

    dims = [d for d in subset_dimensions.keys() if any(r["dimension"] == d for r in rows)]
    labels = []
    for dim in dims:
        subset_labels = [label for label, _ in subset_dimensions[dim]]
        for label in subset_labels:
            if any(r["dimension"] == dim and r["label"] == label for r in rows):
                labels.append(label)

    matrix = pd.DataFrame(index=dims, columns=labels, dtype=float)
    for r in rows:
        matrix.loc[r["dimension"], r["label"]] = r[delta_key]

    fig, ax = plt.subplots(figsize=(max(6, len(labels) * 1.4), max(3, len(dims) * 1.0 + 1)))
    im = ax.imshow(matrix.values.astype(float), cmap=plt.cm.RdBu, aspect="auto", vmin=-0.05, vmax=0.05)

    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(dims)))
    ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
    ax.set_yticklabels(dims, fontsize=9)

    for i, dim in enumerate(dims):
        for j, lbl in enumerate(labels):
            val = matrix.loc[dim, lbl]
            if not pd.isna(val):
                txt_color = "white" if abs(float(val)) > 0.035 else "black"
                ax.text(j, i, f"{float(val):+.3f}", ha="center", va="center",
                        fontsize=8.5, color=txt_color, fontweight="bold")

    plt.colorbar(im, ax=ax, label="ΔHit@10 (Proposed − Baseline)", shrink=0.8)
    ax.set_title("ΔHit@10 Across Subsets and Dimensions", fontsize=13, fontweight="bold", pad=12)

    plt.tight_layout()
    path = os.path.join(save_dir, "fig2_delta_heatmap.png")
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    print(f"[saved] {path}")

def plot_ablation_across_subsets(all_results, save_dir, subset_dimensions):
    ablation_pairs = [
        ("Hit@10 (no_weather→full)",  "Weather removed",  "#C0392B"),
        ("Hit@10 (no_occasion→full)", "Occasion removed", "#E67E22"),
        ("Hit@10 (no_style→full)",    "Style removed",    "#8E44AD"),
    ]

    rows = [
        r for r in all_results
        if r.get("label") not in ("All (union)", "Full dataset")
        and r.get("dimension") != "Overall"
        and r.get("n", 0) > 0
    ]
    rows = sort_result_rows(rows, subset_dimensions)

    if not rows:
        print("[WARN] plot_ablation_across_subsets: no data")
        return

    xlabels = [r["label"] for r in rows]
    x = np.arange(len(xlabels))

    fig, ax = plt.subplots(figsize=(max(9, len(xlabels) * 1.5), 5.5))

    for abl_key, abl_label, color in ablation_pairs:
        delta_key = f"Δ{abl_key}"
        vals = [r.get(delta_key, np.nan) for r in rows]
        ax.plot(x, vals, marker="o", label=abl_label, color=color, linewidth=2, markersize=6)

    ax.axhline(0, color="#999999", linewidth=1, linestyle="--")
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, rotation=25, ha="right", fontsize=8.5)
    ax.set_ylabel("ΔHit@10 (when factor removed vs full)", fontsize=10)
    ax.set_title(
        "Factor Contribution by Subset\n"
        "(Positive = removing this factor hurts more in this subset → factor is more important here)",
        fontsize=11, fontweight="bold", pad=10
    )
    ax.legend(fontsize=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    prev_dim = None
    ymax = ax.get_ylim()[1]
    for i, r in enumerate(rows):
        if r["dimension"] != prev_dim and i > 0:
            ax.axvline(i - 0.5, color="#cccccc", linewidth=1, linestyle="--")
            ax.text(i - 0.45, ymax * 0.96, r["dimension"], ha="left", va="top", fontsize=7.5, color="#888888")
        if prev_dim is None:
            ax.text(i - 0.1, ymax * 0.96, r["dimension"], ha="left", va="top", fontsize=7.5, color="#888888")
        prev_dim = r["dimension"]

    plt.tight_layout()
    path = os.path.join(save_dir, "fig3_ablation_across_subsets.png")
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    print(f"[saved] {path}")

def plot_per_dimension_bars(all_results, save_dir, subset_dimensions):
    metric_key = "Hit@10"
    left_key = f"{metric_key}_left"
    right_key = f"{metric_key}_right"

    dim_names = list(subset_dimensions.keys())
    n_dims = len(dim_names)
    ncols = 2
    nrows = (n_dims + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 6.5, nrows * 4.2))
    axes = axes.flatten() if n_dims > 1 else [axes]

    for idx, dim in enumerate(dim_names):
        ax = axes[idx]
        rows = [
            r for r in all_results
            if r.get("dimension") == dim
            and r.get("label") != "All (union)"
            and r.get("n", 0) > 0
            and left_key in r and right_key in r
        ]
        rows = sort_result_rows(rows, subset_dimensions)

        if not rows:
            ax.set_visible(False)
            continue

        labels = [f"{r['label']}\n(n={r['n']})" for r in rows]
        orig_vals = [r[left_key] for r in rows]
        prop_vals = [r[right_key] for r in rows]
        x = np.arange(len(labels))
        w = 0.35

        b1 = ax.bar(x - w / 2, orig_vals, w, color="#888888", edgecolor="white", label="Original")
        b2 = ax.bar(x + w / 2, prop_vals, w, color="#2166AC", edgecolor="white", label="Proposed")

        for bar in b1:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                    f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)
        for bar in b2:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                    f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8, color="#2166AC")

        ax.set_title(dim, fontsize=11, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=9)
        ax.set_ylabel("Hit@10", fontsize=9)
        ax.set_ylim(0, max(max(orig_vals + [0.01]), max(prop_vals + [0.01])) * 1.20)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        if idx == 0:
            ax.legend(fontsize=9)

    for i in range(n_dims, len(axes)):
        axes[i].set_visible(False)

    fig.suptitle("Hit@10: Original vs Proposed — Per Subset Dimension",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()

    path = os.path.join(save_dir, "fig4_per_dimension_bars.png")
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    print(f"[saved] {path}")

def plot_rank_improvement_by_subset(all_results, save_dir, subset_dimensions):
    metric_key = "Median Rank"
    delta_key = f"Δ{metric_key}"

    rows = [
        r for r in all_results
        if delta_key in r
        and r.get("label") != "All (union)"
        and r.get("dimension") != "Overall"
        and r.get("n", 0) > 0
    ]
    rows = sort_result_rows(rows, subset_dimensions)

    if not rows:
        print("[WARN] plot_rank_improvement_by_subset: no data")
        return

    labels = [f"{r['label']}\n(n={r['n']})" for r in rows]
    deltas = [r[delta_key] for r in rows]
    gain = [-d for d in deltas]
    colors = ["#2166AC" if g >= 0 else "#D6604D" for g in gain]

    fig, ax = plt.subplots(figsize=(max(8, len(labels) * 1.5), 5))
    bars = ax.bar(range(len(labels)), gain, color=colors, edgecolor="white")

    for bar, g in zip(bars, gain):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + (0.3 if g >= 0 else -0.8),
            f"{g:+.1f}",
            ha="center",
            va="bottom" if g >= 0 else "top",
            fontsize=8,
            color="#2166AC" if g >= 0 else "#D6604D"
        )

    ax.axhline(0, color="#999999", linewidth=1, linestyle="--")
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=25, ha="right", fontsize=8.5)
    ax.set_ylabel("Rank Improvement (↑ better)", fontsize=10)
    ax.set_title(
        "Median Rank Improvement (Baseline→Proposed) by Subset\n"
        "(Positive = rank improved = fewer distractors above GT)",
        fontsize=11, fontweight="bold"
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    prev_dim = None
    for i, r in enumerate(rows):
        if r["dimension"] != prev_dim and i > 0:
            ax.axvline(i - 0.5, color="#cccccc", linewidth=1, linestyle="--")
        prev_dim = r["dimension"]

    plt.tight_layout()
    path = os.path.join(save_dir, "fig5_rank_improvement_by_subset.png")
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    print(f"[saved] {path}")


# ──────────────────────────────────────────────────────────
# 9) 儲存 CSV
# ──────────────────────────────────────────────────────────
def save_summary_csv(all_results, save_dir):
    df = pd.DataFrame(all_results)

    path = os.path.join(save_dir, "subset_analysis_summary.csv")
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"[saved] {path}")

    delta_key = "ΔHit@10"
    if delta_key in df.columns:
        pivot = df[df["label"] != "All (union)"].pivot_table(
            index="dimension", columns="label", values=delta_key
        )
        pivot_path = os.path.join(save_dir, "subset_delta_hit10_pivot.csv")
        pivot.to_csv(pivot_path, encoding="utf-8-sig")
        print(f"[saved] {pivot_path}")

        print("\n=== ΔHit@10 Pivot Table ===")
        print(pivot.round(4).to_string())


# ──────────────────────────────────────────────────────────
# 10) 主流程
# ──────────────────────────────────────────────────────────
def main():
    print("=" * 70)
    print("  Subset Analysis — Discussion Section Experiment")
    print("  目的：檢查方法在不同情境下的提升是否一致")
    print("=" * 70)

    # Step 1: 建立測試資料層級的 combined dataframe
    combined, cat_df, temp_threshold = build_combined(VARIANTS)

    # 由 categories.csv 推導真正的衣物 / 配件大類
    all_major, clothing_major_cats, accessory_major_cats = derive_category_groups(cat_df)

    print("\n[Debug] Category groups from categories.csv")
    print(f"  all major categories: {all_major}")
    print(f"  clothing major categories: {clothing_major_cats}")
    print(f"  accessory major categories: {accessory_major_cats}")

    print("\n[Debug] Sample category values")
    for col in ["target_item_fg", "fine_category", "major_category"]:
        if col in combined.columns:
            vals = combined[col].astype(str).drop_duplicates().head(15).tolist()
            print(f"  {col}: {vals}")

    subset_dimensions = build_subset_dimensions(
        temp_threshold=temp_threshold,
        clothing_major_cats=clothing_major_cats,
        accessory_major_cats=accessory_major_cats
    )

    # Step 2: 執行子集分析
    print("\n[Step 2] Running subset analysis ...")
    all_results = run_subset_analysis(combined, subset_dimensions)

    # Step 3: 列印摘要
    print("\n" + "=" * 70)
    print("  RESULTS SUMMARY")
    print("=" * 70)
    print(f"{'Dimension':<32} {'Subset':<28} {'n':>6}  {'Orig Hit@10':>12}  {'Prop Hit@10':>12}  {'ΔHit@10':>10}")
    print("-" * 104)

    summary_rows = sort_result_rows([
        r for r in all_results
        if r.get("label") not in ("All (union)", "Full dataset")
        and r.get("dimension") != "Overall"
    ], subset_dimensions)

    for r in summary_rows:
        orig_h = r.get("Hit@10_left", float("nan"))
        prop_h = r.get("Hit@10_right", float("nan"))
        delta_h = r.get("ΔHit@10", float("nan"))
        arrow = "↑" if pd.notna(delta_h) and delta_h > 0.001 else ("↓" if pd.notna(delta_h) and delta_h < -0.001 else "─")

        print(
            f"  {r.get('dimension',''):<30} {r.get('label',''):<28} {r.get('n',0):>6}  "
            f"{orig_h:>12.4f}  {prop_h:>12.4f}  {delta_h:>+8.4f} {arrow}"
        )

    # Step 4: 產生圖表
    print("\n[Step 4] Generating figures ...")
    plot_hitrate_comparison(all_results, OUTPUT_DIR, subset_dimensions)
    plot_delta_heatmap(all_results, OUTPUT_DIR, subset_dimensions)
    plot_ablation_across_subsets(all_results, OUTPUT_DIR, subset_dimensions)
    plot_per_dimension_bars(all_results, OUTPUT_DIR, subset_dimensions)
    plot_rank_improvement_by_subset(all_results, OUTPUT_DIR, subset_dimensions)

    # Step 5: 儲存 CSV
    print("\n[Step 5] Saving CSV summary ...")
    save_summary_csv(all_results, OUTPUT_DIR)

    # Step 6: Discussion hints
    print("\n" + "=" * 70)
    print("  DISCUSSION HINTS")
    print("=" * 70)

    deltas = [
        (r["label"], r["dimension"], r.get("ΔHit@10", np.nan))
        for r in all_results
        if r.get("label") not in ("All (union)", "Full dataset")
        and r.get("dimension") != "Overall"
        and r.get("n", 0) > 30
        and pd.notna(r.get("ΔHit@10", np.nan))
    ]

    if deltas:
        best = max(deltas, key=lambda x: x[2])
        worst = min(deltas, key=lambda x: x[2])
        consistent = all(d >= 0 for _, _, d in deltas)
        improvement_gap = best[2] - worst[2]

        print(f"  ▶  最大提升子集 : [{best[1]}] {best[0]}  ΔHit@10={best[2]:+.4f}")
        print(f"  ▶  最小提升子集 : [{worst[1]}] {worst[0]}  ΔHit@10={worst[2]:+.4f}")
        print(f"  ▶  所有子集皆非負提升: {'Yes ✓' if consistent else 'No — check negative subsets'}")
        print(
            f"  ▶  最大 - 最小提升差距: {improvement_gap:.4f}  "
            f"({'narrow, consistent improvement' if improvement_gap < 0.03 else 'large gap, method is context-sensitive'})"
        )

    print(f"\n✅ 全部完成。圖表 + CSV 已輸出至：{os.path.abspath(OUTPUT_DIR)}/")


if __name__ == "__main__":
    main()

  Subset Analysis — Discussion Section Experiment
  目的：檢查方法在不同情境下的提升是否一致
Loading detail CSVs ...
  original    : 17160 rows
  full        : 17160 rows
  no_weather  : 17160 rows
  no_occasion : 17160 rows
  no_style    : 17160 rows
  original titles: 68306
  fragments:       35140
  category map:    159
  category metadata matched: 17160 / 17160 rows (100.00%)
  extracted temperature rows: 17160 / 17160
  temperature median threshold: 22.90°C
Combined rows: 17160

[Debug] Category groups from categories.csv
  all major categories: ['all body', 'bottoms', 'tops', 'outerwear', 'bags', 'shoes', 'accessories', 'hats', 'jewellery', 'scarves', 'sunglasses']
  clothing major categories: ['all body', 'bottoms', 'tops', 'outerwear']
  accessory major categories: ['bags', 'shoes', 'accessories', 'hats', 'jewellery', 'scarves', 'sunglasses']

[Debug] Sample category values
  target_item_fg: ['46', '37', '4', '62', '43', '57', '38', '64']
  fine_category: ['heels', 'purse', 'dress', 'necklace', 'p